In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install torch torchvision scikit-learn

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import torchvision.models as models
import numpy as np

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(0.3, 0.3, 0.3),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
train_dir = "/content/drive/MyDrive/RAF-DB/DATASET/train"
test_dir  = "/content/drive/MyDrive/RAF-DB/DATASET/test"

train_ds = ImageFolder(train_dir, transform=train_transform)
test_ds  = ImageFolder(test_dir, transform=test_transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)

In [ ]:
class_counts = np.bincount(train_ds.targets)
class_weights = 1. / class_counts
class_weights = class_weights / class_weights.sum()
class_weights = torch.FloatTensor(class_weights).to(device)

In [ ]:
backbone = models.resnet50(pretrained=True)

num_features = backbone.fc.in_features
backbone.fc = nn.Identity()

model = nn.Sequential(
    backbone,
    nn.Linear(num_features, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 7)
).to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 208MB/s]


In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.1
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=25
)

In [ ]:
scaler = torch.amp.GradScaler('cuda')

for epoch in range(25):
    model.train()
    running_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = model(x)
            loss = criterion(outputs, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    scheduler.step()

    print(f"Epoch [{epoch+1}/25] "
          f"Loss: {running_loss/len(train_loader):.4f}")

Epoch [1/25] Loss: 1.7868
Epoch [2/25] Loss: 1.4545
Epoch [3/25] Loss: 1.3088
Epoch [4/25] Loss: 1.2289
Epoch [5/25] Loss: 1.1689
Epoch [6/25] Loss: 1.0978
Epoch [7/25] Loss: 1.0542
Epoch [8/25] Loss: 1.0198
Epoch [9/25] Loss: 0.9747
Epoch [10/25] Loss: 0.9404
Epoch [11/25] Loss: 0.9056
Epoch [12/25] Loss: 0.8891
Epoch [13/25] Loss: 0.8769
Epoch [14/25] Loss: 0.8510
Epoch [15/25] Loss: 0.8386
Epoch [16/25] Loss: 0.8297
Epoch [17/25] Loss: 0.8152
Epoch [18/25] Loss: 0.8076
Epoch [19/25] Loss: 0.8028
Epoch [20/25] Loss: 0.7999
Epoch [21/25] Loss: 0.7939
Epoch [22/25] Loss: 0.7913
Epoch [23/25] Loss: 0.7882
Epoch [24/25] Loss: 0.7879
Epoch [25/25] Loss: 0.7895


In [ ]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        outputs = model(x)
        preds = outputs.argmax(1)

        y_true.extend(y.numpy())
        y_pred.extend(preds.cpu().numpy())

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted')
recall    = recall_score(y_true, y_pred, average='weighted')
f1        = f1_score(y_true, y_pred, average='weighted')

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))

Accuracy : 0.8527
Precision: 0.8578
Recall   : 0.8527
F1 Score : 0.8545


In [ ]:
import os

base_path = "/content/drive/MyDrive/Facial_emotion_project"
models_path = os.path.join(base_path, "models")

os.makedirs(models_path, exist_ok=True)

print("Folders created:", os.path.exists(models_path))

Folders created: True


In [ ]:
torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/Facial_emotion_project/models/simclr.pth"
)

In [ ]:
import os
os.path.isfile("/content/drive/MyDrive/Facial_emotion_project/models/simclr.pth")

True